# 18. Markov Decision Process (MDP)
**Use case:** Oil & Gas — Pump maintenance policy optimisation

A field operates dozens of pumps that drift from Good to Worn to Critical and eventually fail, stopping production until replaced. Crews either run equipment to failure or service on a fixed calendar, and both waste money - failures cost thousands of barrels while unnecessary repairs cost downtime and parts. Condition-monitoring data and CMMS work orders show how pumps degrade and what each action costs. Success means a simple state-to-action rule that lowers two-year cost and failure weeks versus current practice, while respecting safety limits.

> 💡 Think of looking after a car: in each condition (fine, noisy, smoking) you can keep driving, service it or replace it. Each choice has a cost today and changes the odds of what condition the car is in next month. An MDP works out the rule that keeps your total cost lowest over the years.

Legend in code: `# 🐍 PY:` Python lesson · `# 🧠 ML:` ML lesson

In [ ]:
# Run once (Colab already has most of these)
!pip -q install numpy pandas scikit-learn scipy xgboost torch

## EASY: Simulate a pump wearing out week by week
Define pump health states and a transition matrix, simulate the 'never maintain' strategy, and see how fast failure probability grows.

| 5W + H | |
|---|---|
| **Who** | A reliability engineer at an oilfield operator responsible for electric submersible and transfer pumps. |
| **What** | A Markov chain of pump health (Good, Worn, Critical, Failed) and the cost of just running it. |
| **When** | During maintenance budgeting, to show why doing nothing is expensive. |
| **Where** | A short numpy script on the engineer's laptop. |
| **Why** | Before choosing a maintenance policy you must model how equipment degrades and what each state costs. |
| **How** | Write a 4x4 transition matrix, simulate random 20-week paths, propagate the state distribution with matrix multiplication, and average 1,000 runs. |

**🧠 ML concepts:** States and transitions, Markov property, Transition probability matrix, State distribution propagation, Monte Carlo simulation  
**🐍 Python concepts:** Lists as index-to-name lookups, 2-D numpy arrays and row sums, Dicts for costs, Functions returning tuples, Matrix multiplication with @, Underscore digit separators in numbers

In [ ]:
# Easy: model a pump's health as a Markov chain and simulate "just keep running it"
import numpy as np

np.random.seed(42)
# 🐍 PY: a list of state names - the position in the list is the state number
states = ["Good", "Worn", "Critical", "Failed"]

# 🧠 ML: transition matrix P[i][j] = chance of moving from state i to state j in one week of running
# Each row must add up to 1.0 (the pump has to end up somewhere)
P_run = np.array([
    [0.85, 0.15, 0.00, 0.00],   # from Good
    [0.00, 0.80, 0.18, 0.02],   # from Worn
    [0.00, 0.00, 0.65, 0.35],   # from Critical
    [0.00, 0.00, 0.00, 1.00],   # from Failed (stays failed until someone acts)
])
# 🐍 PY: .sum(axis=1) adds across each row; np.allclose checks all are ~1.0
print("Rows sum to 1:", np.allclose(P_run.sum(axis=1), 1.0))

# 🐍 PY: dict mapping state name -> weekly cost in $ (lost production, extra energy, etc.)
weekly_cost = {"Good": 0, "Worn": 1_000, "Critical": 4_000, "Failed": 25_000}

# 🧠 ML: Markov property - the next state depends only on the current state, not the history
def simulate(weeks=20):
    s = 0  # start in Good
    path, total = [], 0
    for _ in range(weeks):
        # 🐍 PY: np.random.choice picks the next state using row s as probabilities
        s = np.random.choice(4, p=P_run[s])
        path.append(states[s])
        total += weekly_cost[states[s]]
    return path, total

path, cost = simulate()
# 🐍 PY: str.join builds one string; [:1] slices the first letter of each state name
print("One 20-week run:", " ".join(p[:1] for p in path))
print(f"Cost of that run: ${cost:,}")

# 🧠 ML: multiply the start vector by P repeatedly to get the probability of each state after n weeks
dist = np.array([1.0, 0, 0, 0])
for week in range(1, 21):
    dist = dist @ P_run
    if week in (1, 5, 10, 20):
        # 🐍 PY: zip pairs names with probabilities; :.2f formats each to two decimals
        print(f"Week {week:>2}: " + "  ".join(f"{n}={p:.2f}" for n, p in zip(states, dist)))

# Monte Carlo average over many runs
# 🐍 PY: list comprehension runs the simulation 1000 times and keeps only the cost (index [1])
costs = [simulate()[1] for _ in range(1000)]
print(f"\nAverage 20-week cost if we never maintain: ${np.mean(costs):,.0f}")
print("Next step: add actions (Repair, Replace) and let an MDP choose the cheapest policy.")

**Try this next:**
- Change the Worn -> Critical probability from 0.18 to 0.05 and see the average cost drop.
- Extend weeks to 52 and compare the week-52 distribution.
- Add a crude 'repair when Critical' rule inside simulate() by resetting s to 0 and adding $6,000.

## MEDIUM: Learn transitions from logs, solve with value iteration
Estimate transition probabilities for Run, Repair and Replace from a synthetic maintenance log, then use value iteration to find the cheapest action in each state.

| 5W + H | |
|---|---|
| **Who** | A reliability engineering team with a computerised maintenance management system (CMMS). |
| **What** | An MDP with 4 states and 3 actions whose transition probabilities are learned from 4,000 weekly records. |
| **When** | Quarterly, when the maintenance strategy for a pump fleet is reviewed. |
| **Where** | A Python analysis on exported CMMS work orders and condition-monitoring states. |
| **Why** | Engineers argue about when to intervene; an MDP turns costs and degradation data into a provably optimal rule for each condition. |
| **How** | Simulate 40 pumps for 100 weeks, count (state, action, next state) with groupby, normalise to probabilities, build a cost table, and iterate the Bellman equation until values converge. |

**🧠 ML concepts:** MDP components: states, actions, transitions, costs, discount, Maximum-likelihood transition estimation, Bellman optimality equation, Value iteration and convergence, Q-values, Handling unseen state-action pairs  
**🐍 Python concepts:** f-string zero padding, List of dicts to DataFrame, groupby with multiple keys and size(), np.divide with where=, Broadcasting with None axes, np.einsum

In [ ]:
# Medium: learn transition probabilities from maintenance logs, then solve the MDP with value iteration
import numpy as np
import pandas as pd

np.random.seed(42)
STATES = ["Good", "Worn", "Critical", "Failed"]
ACTIONS = ["Run", "Repair", "Replace"]
S, A = len(STATES), len(ACTIONS)

# Hidden "true" physics used only to generate the synthetic log
TRUE_P = {
    "Run":     [[0.85, 0.15, 0, 0], [0, 0.80, 0.18, 0.02], [0, 0, 0.65, 0.35], [0, 0, 0, 1.0]],
    "Repair":  [[0.90, 0.10, 0, 0], [0.70, 0.30, 0, 0], [0.10, 0.60, 0.30, 0], [0, 0.30, 0.50, 0.20]],
    "Replace": [[1.0, 0, 0, 0]] * 4,
}

# 1. Simulate 4000 weekly records from a fleet of 40 pumps (what a CMMS system would export)
rows = []
for pump in range(40):
    s = 0
    for week in range(100):
        # Field crews historically chose actions somewhat randomly - good for learning all transitions
        a = np.random.choice(A, p=[0.70, 0.20, 0.10]) if s < 3 else np.random.choice([1, 2])
        s_next = np.random.choice(S, p=TRUE_P[ACTIONS[a]][s])
        # 🐍 PY: append a dict per row - pandas turns a list of dicts into a table in one call
        rows.append({"pump_id": f"P{pump:02d}", "week": week, "state": STATES[s],
                     "action": ACTIONS[a], "next_state": STATES[s_next]})
        s = s_next
log = pd.DataFrame(rows)
print(f"Maintenance log: {len(log)} rows, {log.pump_id.nunique()} pumps")
print(log.head(4).to_string(index=False), "\n")

# 2. Estimate P(s' | s, a) by counting
# 🧠 ML: maximum-likelihood transition estimate = count(s,a,s') / count(s,a)
P = np.zeros((A, S, S))
# 🐍 PY: groupby on three columns + size() counts how often each (state, action, next) happened
counts = log.groupby(["action", "state", "next_state"]).size()
for (a, s, s2), n in counts.items():
    P[ACTIONS.index(a), STATES.index(s), STATES.index(s2)] = n
row_tot = P.sum(axis=2, keepdims=True)
# 🐍 PY: np.divide with where= avoids divide-by-zero for (state, action) pairs never seen
P = np.divide(P, row_tot, out=np.zeros_like(P), where=row_tot > 0)
# 🧠 ML: never-seen pairs (e.g. Run while Failed) get a conservative assumption - the pump stays put
for a in range(A):
    for s in range(S):
        if row_tot[a, s, 0] == 0:
            P[a, s, s] = 1.0
            print(f"No data for ({STATES[s]}, {ACTIONS[a]}) -> assume it stays {STATES[s]}")
print("Estimated P(next | state, Run):")
print(pd.DataFrame(P[0], index=STATES, columns=STATES).round(2).to_string(), "\n")

# 3. Costs per week in $ (we minimise cost, so reward = -cost)
operating = np.array([0, 1_000, 4_000, 25_000])        # lost output by state
action_cost = np.array([0, 6_000, 20_000])              # Run, Repair, Replace
# 🐍 PY: broadcasting - (S,1) + (A,) -> an S x A cost table in one line
C = operating[:, None] + action_cost[None, :]

# 4. Value iteration
# 🧠 ML: Bellman update V(s) = min_a [ cost(s,a) + gamma * sum_s' P(s'|s,a) * V(s') ]
gamma, V = 0.95, np.zeros(S)
for it in range(1, 1000):
    # 🐍 PY: einsum "ask,k->as" multiplies each P[a,s,:] by V and sums over next states
    Q = C.T + gamma * np.einsum("ask,k->as", P, V)
    V_new = Q.min(axis=0)
    # 🧠 ML: stop when values barely change - the policy has converged
    if np.max(np.abs(V_new - V)) < 1e-3:
        break
    V = V_new
print(f"Value iteration converged in {it} sweeps (gamma={gamma})")
policy = Q.argmin(axis=0)
print("\nState     | Best action | Expected discounted cost")
for s in range(S):
    print(f"{STATES[s]:<9} | {ACTIONS[policy[s]]:<11} | ${V[s]:>10,.0f}")

# 5. Business interpretation
print("\nQ-table ($, lower is better):")
print(pd.DataFrame(Q.T, index=STATES, columns=ACTIONS).round(0).astype(int).to_string())
print("\nReading: keep running healthy pumps, intervene before failure - the policy shows exactly"
      " where paying for maintenance beats risking a $25k/week outage.")

**Try this next:**
- Lower gamma to 0.5 and see whether the policy stops repairing Worn pumps.
- Raise the Repair cost to 15,000 and rerun.
- Shrink the log to 5 pumps and check how noisy the estimated transition matrix becomes.

## HARD: Policy vs value iteration with oil-price regimes and baselines
Solve an 8-state pump MDP (health x oil price) with both policy and value iteration, test discount sensitivity, and prove the policy beats run-to-failure and fixed-schedule maintenance in Monte Carlo.

| 5W + H | |
|---|---|
| **Who** | Asset-integrity and production engineering leads deciding maintenance strategy for a pump fleet. |
| **What** | A richer MDP whose costs depend on both pump health and the current oil price regime, solved and stress-tested. |
| **When** | Annual maintenance strategy review and whenever oil price or repair-cost assumptions change materially. |
| **Where** | A decision-support model whose policy table is published to the CMMS and field supervisors. |
| **Why** | Lost barrels cost more when prices are high, and downtime during repairs matters; leadership needs evidence the policy beats current practice. |
| **How** | Build joint transitions with a Kronecker product, solve with value and policy iteration, sweep gamma, simulate 1,000 pumps x 104 weeks for three strategies, test a repair-cost shock and save the policy. |

**🧠 ML concepts:** Factored state spaces (health x price), Policy iteration: evaluation by linear solve + greedy improvement, Value iteration vs policy iteration convergence, Discount factor as planning horizon, Monte Carlo policy evaluation against baselines, Sensitivity analysis of the optimal policy  
**🐍 Python concepts:** Classes with __init__ and methods, Nested list comprehension for names, np.kron and np.stack, Fancy indexing P[policy, arange], Lambdas with a shared signature, time.perf_counter for benchmarking

In [ ]:
# Hard: pump maintenance MDP with price regimes - policy vs value iteration, discount sensitivity,
# and Monte Carlo comparison against run-to-failure and fixed-schedule baselines
import os
import time
import tempfile
import joblib
import numpy as np

HEALTH = ["Good", "Worn", "Critical", "Failed"]
PRICE = ["LowPrice", "HighPrice"]
ACTIONS = ["Run", "Repair", "Replace"]
# 🐍 PY: nested comprehension builds the 8 combined state names like "Worn/HighPrice"
STATE_NAMES = [f"{h}/{p}" for p in PRICE for h in HEALTH]
H, NP, A = len(HEALTH), len(PRICE), len(ACTIONS)
S = H * NP


class PumpMDP:
    """Health x oil-price-regime MDP. Cost = lost production + maintenance spend."""

    def __init__(self, repair_cost=6_000, replace_cost=20_000):
        health_P = np.array([
            [[0.85, 0.15, 0, 0], [0, 0.80, 0.18, 0.02], [0, 0, 0.65, 0.35], [0, 0, 0, 1.0]],     # Run
            [[0.90, 0.10, 0, 0], [0.70, 0.30, 0, 0], [0.10, 0.60, 0.30, 0], [0, 0.30, 0.50, 0.20]],  # Repair
            [[1.0, 0, 0, 0]] * 4,                                                               # Replace
        ])
        # 🧠 ML: the price regime evolves on its own (exogenous), independent of what we do to the pump
        price_P = np.array([[0.95, 0.05], [0.08, 0.92]])
        # 🧠 ML: joint transition = health transition x price transition (Kronecker product)
        self.P = np.stack([np.kron(price_P, health_P[a]) for a in range(A)])
        lost_bbl = np.array([0, 40, 160, 1000])          # barrels lost per week by health
        oil_price = np.array([45, 85])                    # $/bbl by regime
        # 🧠 ML: production loss also happens while the crew repairs/replaces (downtime)
        downtime_bbl = np.array([0, 150, 400])
        # 🐍 PY: index p * H + h flattens (price, health) into one state number
        self.C = np.zeros((S, A))
        for p in range(NP):
            for h in range(H):
                for a in range(A):
                    self.C[p * H + h, a] = ((lost_bbl[h] + downtime_bbl[a]) * oil_price[p]
                                            + [0, repair_cost, replace_cost][a])

    def q_values(self, V, gamma):
        # 🐍 PY: einsum sums P[a,s,k] * V[k] over k, giving an (A,S) array
        return self.C.T + gamma * np.einsum("ask,k->as", self.P, V)


def value_iteration(mdp, gamma, tol=1e-4):
    # 🧠 ML: start from zero values; each sweep looks one more week into the future
    V, sweeps = np.zeros(S), 0
    while True:
        sweeps += 1
        V_new = mdp.q_values(V, gamma).min(axis=0)
        if np.max(np.abs(V_new - V)) < tol:
            return mdp.q_values(V_new, gamma).argmin(axis=0), V_new, sweeps
        V = V_new


def policy_iteration(mdp, gamma):
    policy, rounds = np.zeros(S, dtype=int), 0
    while True:
        rounds += 1
        # 🧠 ML: policy evaluation solves the linear system V = C_pi + gamma * P_pi V exactly
        # 🐍 PY: fancy indexing picks, for every state, the transition row of the action the policy chooses
        P_pi = mdp.P[policy, np.arange(S)]
        C_pi = mdp.C[np.arange(S), policy]
        V = np.linalg.solve(np.eye(S) - gamma * P_pi, C_pi)
        # 🧠 ML: policy improvement - switch each state to its greedy action
        new_policy = mdp.q_values(V, gamma).argmin(axis=0)
        if np.array_equal(new_policy, policy):
            return policy, V, rounds
        policy = new_policy


def simulate(mdp, policy_fn, weeks=104, runs=1000, seed=42):
    """Monte Carlo: average undiscounted 2-year cost of a decision rule."""
    rng = np.random.default_rng(seed)
    totals, failures = np.zeros(runs), np.zeros(runs)
    for r in range(runs):
        s = 0
        for t in range(weeks):
            # 🧠 ML: the simulator is policy-agnostic - any rule with signature (state, week) can be evaluated
            a = policy_fn(s, t)
            totals[r] += mdp.C[s, a]
            failures[r] += (s % H == 3)
            # 🧠 ML: sample the next state from the model - this is Monte Carlo evaluation
            s = rng.choice(S, p=mdp.P[a, s])
    return totals.mean(), totals.std(), failures.mean()


mdp = PumpMDP()
gamma = 0.97

# 1. Compare the two solvers
# 🐍 PY: perf_counter is a high-resolution clock for timing code
t0 = time.perf_counter()
pol_vi, V_vi, sweeps = value_iteration(mdp, gamma)
t_vi = time.perf_counter() - t0
t0 = time.perf_counter()
pol_pi, V_pi, rounds = policy_iteration(mdp, gamma)
t_pi = time.perf_counter() - t0
print(f"Value iteration : {sweeps:>4} sweeps  {t_vi * 1000:6.1f} ms")
print(f"Policy iteration: {rounds:>4} rounds  {t_pi * 1000:6.1f} ms")
print(f"Same policy? {np.array_equal(pol_vi, pol_pi)}   max value gap ${np.max(np.abs(V_vi - V_pi)):.2f}\n")
print("State               | Optimal action | Discounted cost")
for s in range(S):
    print(f"{STATE_NAMES[s]:<19} | {ACTIONS[pol_pi[s]]:<14} | ${V_pi[s]:>10,.0f}")

# 2. Discount sensitivity
# 🧠 ML: gamma near 1 = long-term planner; small gamma = short-sighted manager who avoids spend today
SHORT = ["Run", "Fix", "New"]  # Run / Repair / Replace
print("\nGamma | Policy (LowPrice: G W C F | HighPrice: G W C F)")
for g in [0.50, 0.80, 0.90, 0.97, 0.99]:
    pol, _, _ = policy_iteration(mdp, g)
    # 🐍 PY: slicing pol[:H] and pol[H:] splits the policy into the two price regimes
    lo = " ".join(SHORT[a] for a in pol[:H])
    hi = " ".join(SHORT[a] for a in pol[H:])
    print(f" {g:.2f} | {lo}  |  {hi}")

# 3. Monte Carlo evaluation vs baselines
# 🐍 PY: lambdas make small decision rules with the same (state, week) signature
baselines = {
    "Run to failure": lambda s, t: 2 if s % H == 3 else 0,
    "Fixed schedule (8wk)": lambda s, t: 2 if s % H == 3 else (1 if t % 8 == 7 else 0),
    "MDP optimal policy": lambda s, t: int(pol_pi[s]),
}
print("\nMonte Carlo, 1000 pumps x 104 weeks:")
print("Strategy               | Avg 2-yr cost | Std dev   | Failed weeks")
results = {}
for name, fn in baselines.items():
    mean, sd, fails = simulate(mdp, fn)
    results[name] = mean
    print(f"{name:<22} | ${mean:>11,.0f} | ${sd:>8,.0f} | {fails:5.2f}")
saving = results["Run to failure"] - results["MDP optimal policy"]
print(f"\nMDP policy saves ${saving:,.0f} per pump over 2 years vs run-to-failure (synthetic costs).")

# 4. Robustness: what if repair cost doubles?
# 🧠 ML: sensitivity test - which decisions flip if a cost assumption is wrong?
alt_pol, _, _ = policy_iteration(PumpMDP(repair_cost=12_000), gamma)
changed = [STATE_NAMES[s] for s in range(S) if alt_pol[s] != pol_pi[s]]
print(f"If repair cost doubles, policy changes in: {changed or 'no states'}")

path = os.path.join(tempfile.mkdtemp(), "pump_policy.joblib")
joblib.dump({"states": STATE_NAMES, "actions": ACTIONS, "policy": pol_pi, "gamma": gamma}, path)
print(f"Saved policy -> {os.path.basename(path)}")
print("Safety note: a cost-optimal policy must never override API/OSHA/PHMSA safety limits -"
      " Critical units on sour or high-pressure service need mandatory shutdown rules on top.")

**Try this next:**
- Make oil price regimes more persistent (0.99 / 0.99) and see if the policy differs between regimes.
- Add a fourth action 'Inspect' that costs $1,000 and slightly lowers failure odds.
- Change the fixed schedule to every 4 weeks and compare its cost with the MDP policy.